In [1]:
import polars as pl

import nwec.utility_reporting.arrearage_counts
import nwec.utility_reporting.arrearages
import nwec.utils.excel
from nwec.constants import RAW_UTILITY_DATA, Utility

YEAR = 2024
QUARTER = 4
NUM_MONTHS = 12
COLS_PER_MONTH = 1
SHEET_SEARCH_STRING = "past due balances"
ARREARAGE_SEARCH_STRING = "Number of Customers by Customer Class With Past-due balances"
spreadsheet = RAW_UTILITY_DATA / str(YEAR) / f"{Utility.CNG.code}_{YEAR}_Q{QUARTER}.xlsx"
source_date_format = "%Y-%m-%d %H:%M:%S"

In [2]:
sheet_index = nwec.utils.excel.get_sheet_index_from_name(spreadsheet, SHEET_SEARCH_STRING)
df = pl.read_excel(spreadsheet, sheet_id=sheet_index, has_header=False)
arrearage_counts = nwec.utility_reporting.arrearages.get_arrearages_df(
    df, NUM_MONTHS, COLS_PER_MONTH, ARREARAGE_SEARCH_STRING
)

# Arrearage Counts


In [3]:
arrearage_counts

column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,column_13,column_14
str,str,str,str,str,str,str,str,str,str,str,str
null,null,null,null,null,null,null,null,null,null,null,null
null,null,null,null,null,null,null,null,null,null,null,null
"""2024-01-31 00:00:00""","""2024-02-28 00:00:00""","""2024-03-31 00:00:00""","""2024-04-30 00:00:00""","""2024-05-31 00:00:00""","""2024-06-30 00:00:00""","""2024-07-31 00:00:00""","""2024-08-31 00:00:00""","""2024-09-30 00:00:00""","""2024-10-31 00:00:00""","""2024-11-30 00:00:00""","""2024-12-31 00:00:00"""
"""1""","""1""","""1""","""1""","""1""","""2""","""2""","""1""","""1""",null,"""4""",null
"""2""","""2""","""4""","""4""","""1""","""1""","""3""","""2""","""1""","""2""","""17""","""17"""
…,…,…,…,…,…,…,…,…,…,…,…
"""16""","""6""","""20""","""15""","""20""","""19""","""13""","""13""","""9""","""5""","""15""","""11"""
"""39""","""46""","""45""","""40""","""43""","""33""","""43""","""30""","""39""","""33""","""32""","""33"""
"""3""","""1""","""2""","""2""","""3""","""2""","""7""","""3""","""9""","""5""","""5""","""3"""


In [4]:
date_row = nwec.utility_reporting.arrearages.infer_date_row(arrearage_counts, source_date_format)
arrearage_counts = arrearage_counts.tail(-date_row)  # remove rows before the date row
arrearage_counts = nwec.utility_reporting.arrearage_counts.format_arrearage_count_dates(
    arrearage_counts, source_date_format
)
arrearage_counts = nwec.utility_reporting.arrearages.add_zip_and_customer_class_cols(df, arrearage_counts)
arrearage_counts = nwec.utility_reporting.arrearage_counts.normalize_arrearage_count_cols(arrearage_counts, Utility.CNG)


/home/peter/coding/nwec/nwec/utility_reporting/arrearages/arrearages.py:123: UserWarning: Multiple columns have at least 5 rows that match the ZIP code pattern; using the first.
  zip_column = spreadsheet_df.select(pl.nth(nwec.utils.excel.infer_zip_column(spreadsheet_df)))
/home/peter/coding/nwec/nwec/utility_reporting/arrearages/arrearages.py:127: UserWarning: Multiple columns have at least 5 rows that match the customer class pattern; using the first.
  customer_class_column = spreadsheet_df.select(pl.nth(infer_customer_class_column(spreadsheet_df)))


# Save Results


In [5]:
nwec.utility_reporting.arrearage_counts.save_processed_arrearage_counts(arrearage_counts)